# Module B1 & B2 - Customer Review Sentiment Analysis

This notebook trains a Natural Language Processing (NLP) model to classify customer reviews into Positive, Neutral, or Negative sentiment.

It is configured to run either locally (using a generated mock reviews dataset) or in Kaggle using a sampled subset of the real Amazon Fine Food Reviews dataset.

In [ ]:
import os
import re
import glob
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

In [ ]:
# Auto-detect Kaggle environment
KAGGLING = os.path.exists('/kaggle')

if KAGGLING:
    print("Running in Kaggle environment!")
    # Case-insensitive recursive scan for reviews CSV file
    input_files = []
    if os.path.exists('/kaggle/input'):
        for root, dirs, files in os.walk('/kaggle/input'):
            for f in files:
                f_lower = f.lower()
                if f_lower == 'reviews.csv' or 'review' in f_lower:
                    input_files.append(os.path.join(root, f))
                    
    if input_files:
        REVIEWS_CSV_PATH = input_files[0]
    else:
        REVIEWS_CSV_PATH = "/kaggle/input/amazon-fine-food-reviews/Reviews.csv"
    print(f"Kaggle Reviews Dataset Path: {REVIEWS_CSV_PATH}")
    MODEL_DIR = "/kaggle/working/models"
    DATA_DIR = "/kaggle/working/data"
else:
    print("Running locally.")
    REVIEWS_CSV_PATH = os.path.abspath(os.path.join(os.getcwd(), '..', 'data', 'reviews.csv'))
    DATA_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', 'data'))
    MODEL_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', 'app', 'models'))

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

In [ ]:
# Load or Generate Sentiment Dataset
if KAGGLING and os.path.exists(REVIEWS_CSV_PATH):
    print("Loading and parsing real Amazon Reviews dataset...")
    # Load with only required columns to save memory
    df_full = pd.read_csv(REVIEWS_CSV_PATH, usecols=['Score', 'Text'])
    
    # Map Score rating to Sentiment classes
    def map_score_to_sentiment(score):
        if score >= 4:
            return 'positive'
        elif score == 3:
            return 'neutral'
        else:
            return 'negative'
            
    df_full['sentiment'] = df_full['Score'].apply(map_score_to_sentiment)
    df_full = df_full.rename(columns={'Text': 'review_text'})
    
    # Sample a balanced subset of 10k reviews per class to ensure fast execution
    g = df_full.groupby('sentiment')
    df = g.apply(lambda x: x.sample(min(len(x), 10000), random_state=42)).reset_index(drop=True)
    print(f"Sampled balanced dataset of size: {df.shape}")
else:
    # Local fallback or mock generator
    if not os.path.exists(REVIEWS_CSV_PATH):
        print("Local reviews.csv not found. Generating a mock dataset...")
        mock_reviews = [
            ("Absolutely love this store! The staff is super friendly.", "positive"),
            ("Great product quality. Shipped extremely fast.", "positive"),
            ("Amazing service, exceeded my expectations!", "positive"),
            ("Highly recommend these shoes, very comfortable.", "positive"),
            ("Perfect purchase. Will buy from this retailer again.", "positive"),
            ("The packaging was neat, and the item arrived in pristine condition.", "positive"),
            ("Best online shopping experience I've had in a long time!", "positive"),
            ("Excellent customer support. They helped me track my package.", "positive"),
            ("Fits perfectly and the material feels premium.", "positive"),
            ("Super happy with the discount. Exceptional value for money.", "positive"),
            ("Horrible customer service. Nobody replies to my messages.", "negative"),
            ("The item arrived broken. Very disappointed.", "negative"),
            ("Terrible delivery time. Took three weeks to ship!", "negative"),
            ("Quality is cheap, not worth the price at all.", "negative"),
            ("Wrong size sent, and the return process is too complex.", "negative"),
            ("The color does not match the website image. Returning it.", "negative"),
            ("Avoid this brand. The product stopped working after two days.", "negative"),
            ("Waste of money. Extremely bad experience.", "negative"),
            ("Very bad fabric quality. It shrunk after the first wash.", "negative"),
            ("The instructions are confusing and parts were missing.", "negative"),
            ("The item is okay, does what it says. Nothing special.", "neutral"),
            ("Average quality. Decent for the price.", "neutral"),
            ("Shipping took a bit longer than expected, but product is fine.", "neutral"),
            ("It fits fine, but the material is a bit stiff.", "neutral"),
            ("Not bad, but I have seen better products elsewhere.", "neutral"),
            ("Satisfactory experience, although customer service was slow.", "neutral"),
            ("Standard product, standard shipping. Fine overall.", "neutral"),
            ("It's decent. Not great, but not terrible either.", "neutral"),
            ("The size is slightly off, but it still works.", "neutral"),
            ("A standard retail purchase. No issues, but no wow factor.", "neutral")
        ] * 10
        df = pd.DataFrame(mock_reviews, columns=['review_text', 'sentiment'])
        df.to_csv(REVIEWS_CSV_PATH, index=False)
        print(f"Mock dataset written successfully to: {REVIEWS_CSV_PATH}")
    else:
        print(f"Loading existing local reviews.csv from {REVIEWS_CSV_PATH}")
        df = pd.read_csv(REVIEWS_CSV_PATH)

In [ ]:
# Preprocess Review Text
STOPWORDS = {"is", "an", "the", "a", "and", "to", "in", "of", "for", "on", "with", "at", "by", "this", "it", "that"}

def clean_text(text):
    if not isinstance(text, str): 
        return ""
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    words = text.split()
    cleaned_words = [w for w in words if w not in STOPWORDS]
    return " ".join(cleaned_words)

df['cleaned_text'] = df['review_text'].apply(clean_text)
print("Sample Cleanup Result:")
print(" Original:", df['review_text'].iloc[0])
print(" Cleaned :", df['cleaned_text'].iloc[0])

In [ ]:
# Split Data & Apply TF-IDF Feature Extraction
X = df['cleaned_text']
y = df['sentiment']

# Perform split first to avoid any data leakage
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

vectorizer = TfidfVectorizer(max_features=2500, ngram_range=(1, 2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(f"Train TF-IDF shape: {X_train_tfidf.shape}")
print(f"Test TF-IDF shape: {X_test_tfidf.shape}")

In [ ]:
# Train Classifier
model = LogisticRegression(class_weight='balanced', max_iter=1000)
model.fit(X_train_tfidf, y_train)
print("Classifier training completed successfully!")

In [ ]:
# Evaluate Performance (Realistic Non-100% metrics on real data)
y_pred = model.predict(X_test_tfidf)

print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred, labels=model.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=model.classes_)
disp.plot(cmap=plt.cm.Blues)
plt.title("Confusion Matrix - Sentiment Classifier")
plt.show()

In [ ]:
# Serialize Model
model_path = os.path.join(MODEL_DIR, 'sentiment_model.pkl')
vectorizer_path = os.path.join(MODEL_DIR, 'vectorizer.pkl')

with open(model_path, 'wb') as f:
    pickle.dump(model, f)

with open(vectorizer_path, 'wb') as f:
    pickle.dump(vectorizer, f)

print(f"Model saved to: {model_path}")
print(f"Vectorizer saved to: {vectorizer_path}")